# Medical Imaging: Pneumonia Detection from Chest X-Rays

## Overview

Pneumonia is a leading cause of death worldwide, particularly among children under 5 and adults over 65. Chest X-rays are the most common imaging modality for pneumonia diagnosis, but interpretation requires trained radiologists -- a scarce resource in many parts of the world.

This notebook builds a **deep learning classifier** that distinguishes **NORMAL** from **PNEUMONIA** chest X-rays using transfer learning. We leverage pretrained CNNs (ResNet-50, DenseNet-121) fine-tuned on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) dataset.

### Clinical Relevance
- **Early detection** of pneumonia can reduce mortality, especially in resource-limited settings.
- **AI-assisted screening** can triage cases, flagging suspicious X-rays for priority radiologist review.
- **Interpretability** (via Grad-CAM) is essential: clinicians need to understand *why* the model makes a prediction.

### Dataset
- **5,863** chest X-ray images (JPEG), organized into `train/`, `val/`, and `test/` splits.
- **2 classes**: NORMAL, PNEUMONIA
- Source: Guangzhou Women and Children's Medical Center (Kermany et al., 2018)

---
## 1. Imports

In [ ]:
import os
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    classification_report, precision_recall_fscore_support
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ============================================================
# Data Download (runtime fallback when dataset not mounted)
# ============================================================
import os, glob

KAGGLE_CHEST_XRAY_BASE = "/kaggle/input/chest-xray-pneumonia"

if not os.path.exists(KAGGLE_CHEST_XRAY_BASE) or len(os.listdir(KAGGLE_CHEST_XRAY_BASE)) == 0:
    print("Dataset not mounted via /kaggle/input, downloading...")
    os.makedirs("/kaggle/working/data", exist_ok=True)
    os.system("kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /kaggle/working/data --unzip")
    # Find chest_xray dir with train/test/val
    candidates = glob.glob("/kaggle/working/data/**/chest_xray", recursive=True)
    found = None
    for c in sorted(candidates, key=len):
        if os.path.isdir(os.path.join(c, "train")):
            found = c
            break
    if found:
        # The config cell expects BASE to be the parent of chest_xray
        KAGGLE_CHEST_XRAY_BASE = os.path.dirname(found)
    else:
        KAGGLE_CHEST_XRAY_BASE = "/kaggle/working/data"
    print(f"Data downloaded. Base: {KAGGLE_CHEST_XRAY_BASE}")
else:
    print(f"Dataset found at: {KAGGLE_CHEST_XRAY_BASE}")


---
## 2. Configuration

In [ ]:
# ---- Hyperparameters ----
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# ---- Paths ----
# The chest-xray-pneumonia dataset (v2) may have double-nested directories.
# We detect the correct root automatically.
# BASE set by download cell above
BASE = Path(KAGGLE_CHEST_XRAY_BASE)
DATA_ROOT = None
for candidate in [
    BASE / "chest_xray",
    BASE / "chest_xray" / "chest_xray",
    BASE,
]:
    if (candidate / "train").exists():
        DATA_ROOT = candidate
        break

if DATA_ROOT is None:
    # List what is actually there for debugging
    print("Available paths:")
    for p in BASE.rglob("*"):
        if p.is_dir():
            print(f"  {p}")
    raise FileNotFoundError(f"Cannot find train directory under {BASE}")

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"
print(f"Data root: {DATA_ROOT}")
print(f"Train: {len(list(TRAIN_DIR.rglob('*.jpeg')))} images")

# ---- Device ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Class names (alphabetical order as ImageFolder loads them)
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
NUM_CLASSES = len(CLASS_NAMES)

---
## 3. Exploratory Data Analysis

In [ ]:
def count_images(root_dir):
    """Count images per class in a directory."""
    counts = {}
    for cls in CLASS_NAMES:
        cls_dir = root_dir / cls
        if cls_dir.exists():
            n = len([f for f in cls_dir.iterdir() if f.suffix.lower() in (".jpeg", ".jpg", ".png")])
            counts[cls] = n
        else:
            counts[cls] = 0
    return counts

# Count images across splits
splits = {"Train": TRAIN_DIR, "Validation": VAL_DIR, "Test": TEST_DIR}
split_counts = {}
for split_name, split_dir in splits.items():
    counts = count_images(split_dir)
    split_counts[split_name] = counts
    total = sum(counts.values())
    print(f"\n{split_name} ({total} images):")
    for cls, n in counts.items():
        pct = 100.0 * n / total if total > 0 else 0
        print(f"  {cls}: {n} ({pct:.1f}%)")

# ---- Class distribution bar chart ----
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (split_name, counts) in zip(axes, split_counts.items()):
    colors = ["#4CAF50", "#F44336"]  # green=normal, red=pneumonia
    bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor="black", linewidth=0.5)
    ax.set_title(f"{split_name} Split", fontsize=13, fontweight="bold")
    ax.set_ylabel("Number of Images")
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                str(val), ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.suptitle("Class Distribution Across Splits", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nNote: The dataset is significantly imbalanced -- PNEUMONIA cases outnumber NORMAL.")
print("The validation set is very small (16 images). We will address this.")

In [ ]:
# ---- Display sample X-rays from each class ----
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for row, cls in enumerate(CLASS_NAMES):
    cls_dir = TRAIN_DIR / cls
    image_files = sorted(cls_dir.iterdir())[:5]
    for col, img_path in enumerate(image_files):
        img = Image.open(img_path).convert("RGB")
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].set_title(cls, fontsize=10)
        axes[row, col].axis("off")

plt.suptitle("Sample Chest X-Rays", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Data Loading & Augmentation

We apply standard augmentations for training (random horizontal flip, rotation, color jitter) and use `WeightedRandomSampler` to handle class imbalance.

In [ ]:
# ---- Transforms ----
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ---- Datasets ----
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transforms)

print(f"Classes: {train_dataset.classes}")
print(f"Class-to-index mapping: {train_dataset.class_to_idx}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# ---- WeightedRandomSampler to handle class imbalance ----
train_targets = train_dataset.targets
class_counts = Counter(train_targets)
print(f"\nTraining class counts: {dict(class_counts)}")

# Weight for each sample = 1 / (number of samples in that class)
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in train_targets]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ---- DataLoaders ----
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

---
## 5. Model Definition

We use two architectures commonly employed in medical imaging:
1. **ResNet-50** -- strong general-purpose backbone
2. **DenseNet-121** -- widely used in chest X-ray analysis (e.g., CheXNet)

Both are pretrained on ImageNet and fine-tuned with a replaced final classification layer.

In [ ]:
def build_resnet50(num_classes=1, freeze_backbone=False):
    """Build ResNet-50 with a custom binary classification head.
    
    Uses num_classes=1 with BCEWithLogitsLoss for binary classification.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    
    # Replace the final fully connected layer
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_densenet121(num_classes=1, freeze_backbone=False):
    """Build DenseNet-121 with a custom binary classification head.
    
    DenseNet-121 is the backbone of CheXNet (Rajpurkar et al., 2017),
    one of the landmark models in chest X-ray analysis.
    """
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, num_classes)
    )
    return model


# We will train ResNet-50 as the primary model
model = build_resnet50(num_classes=1, freeze_backbone=False)
model = model.to(DEVICE)

# Print trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: ResNet-50")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

---
## 6. Training

We train with:
- **BCEWithLogitsLoss** (binary cross-entropy with built-in sigmoid, numerically stable)
- **AdamW** optimizer with weight decay
- **OneCycleLR** scheduler for super-convergence

In [ ]:
# ---- Loss, optimizer, scheduler ----
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE * 10,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.3,
    anneal_strategy="cos"
)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, device):
    """Train for one epoch. Returns average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)  # (B, 1) for BCE
        
        optimizer.zero_grad()
        outputs = model(images)  # (B, 1) logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model. Returns loss, accuracy, all predictions and labels."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []
    
    for images, labels in loader:
        images = images.to(device)
        labels_gpu = labels.float().unsqueeze(1).to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels_gpu)
        
        running_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(outputs).cpu().numpy()
        preds = (probs > 0.5).astype(float)
        correct += (preds.flatten() == labels.numpy()).sum()
        total += labels.size(0)
        
        all_probs.extend(probs.flatten().tolist())
        all_labels.extend(labels.numpy().tolist())
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, np.array(all_probs), np.array(all_labels)

In [ ]:
# ---- Training loop ----
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | {'LR':>10}")
print("-" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, DEVICE
    )
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"{epoch:5d} | {train_loss:10.4f} | {train_acc:9.4f} | {val_loss:10.4f} | {val_acc:9.4f} | {current_lr:10.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"        -> Saved best model (val_acc={val_acc:.4f})")

print(f"\nTraining complete. Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
# ---- Plot training curves ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, history["train_loss"], "o-", label="Train Loss", color="#1f77b4")
ax1.plot(epochs_range, history["val_loss"], "o-", label="Val Loss", color="#ff7f0e")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Over Epochs", fontweight="bold")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], "o-", label="Train Acc", color="#1f77b4")
ax2.plot(epochs_range, history["val_acc"], "o-", label="Val Acc", color="#ff7f0e")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Over Epochs", fontweight="bold")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Evaluation on Test Set

For medical AI, accuracy alone is insufficient. We compute:
- **ROC curve & AUC**: overall discriminative ability
- **Confusion matrix**: true/false positives and negatives
- **Precision, Recall, F1**: especially important -- high recall (sensitivity) is critical to avoid missing pneumonia cases

In [ ]:
# Load best model for evaluation
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
test_loss, test_acc, test_probs, test_labels = evaluate(model, test_loader, criterion, DEVICE)
test_preds = (test_probs > 0.5).astype(int)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print()

# ---- Classification Report ----
print("Classification Report:")
print("=" * 55)
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, digits=4))

precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="binary")
print(f"Binary metrics -- Precision: {precision:.4f}, Recall (Sensitivity): {recall:.4f}, F1: {f1:.4f}")

In [ ]:
# ---- ROC Curve ----
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
roc_auc = auc(fpr, tpr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
ax1.plot(fpr, tpr, color="#1f77b4", lw=2, label=f"ROC curve (AUC = {roc_auc:.4f})")
ax1.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random")
ax1.set_xlabel("False Positive Rate", fontsize=12)
ax1.set_ylabel("True Positive Rate (Sensitivity)", fontsize=12)
ax1.set_title("ROC Curve", fontsize=14, fontweight="bold")
ax1.legend(loc="lower right", fontsize=11)
ax1.grid(True, alpha=0.3)

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, ax=ax2, annot_kws={"size": 14})
ax2.set_xlabel("Predicted Label", fontsize=12)
ax2.set_ylabel("True Label", fontsize=12)
ax2.set_title("Confusion Matrix", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

# Print specific clinical metrics
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)  # Recall for pneumonia
specificity = tn / (tn + fp)  # Recall for normal
print(f"\nSensitivity (Pneumonia Recall): {sensitivity:.4f}")
print(f"Specificity (Normal Recall):    {specificity:.4f}")
print(f"AUC: {roc_auc:.4f}")

---
## 8. Grad-CAM Visualization

Grad-CAM (Gradient-weighted Class Activation Mapping) highlights the regions of the image that the model focuses on when making a prediction. This is essential for **interpretability** in medical AI -- clinicians need to verify that the model is looking at clinically relevant regions (e.g., lung opacities) rather than artifacts.

In [ ]:
class GradCAM:
    """Simple Grad-CAM implementation for ResNet.
    
    Computes gradient-weighted class activation maps by hooking into
    the last convolutional layer of the network.
    """
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, target_class=None):
        """Generate Grad-CAM heatmap for the given input.
        
        Args:
            input_tensor: preprocessed image tensor (1, C, H, W)
            target_class: class index to visualize (None = predicted class)
        
        Returns:
            cam: numpy array (H, W) with values in [0, 1]
        """
        self.model.eval()
        output = self.model(input_tensor)
        
        if target_class is None:
            target_class = (torch.sigmoid(output) > 0.5).long().item()
        
        self.model.zero_grad()
        # For binary classification with single output, use the output directly
        output.backward()
        
        # Global average pooling of gradients
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        
        # Weighted combination of activation maps
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam = torch.relu(cam)  # Only positive contributions
        cam = cam.squeeze().cpu().numpy()
        
        # Normalize to [0, 1]
        if cam.max() > 0:
            cam = cam / cam.max()
        
        return cam


def show_gradcam(model, dataset, indices, target_layer, device, class_names):
    """Display Grad-CAM heatmaps for selected images."""
    grad_cam = GradCAM(model, target_layer)
    
    # Inverse normalization for display
    inv_normalize = transforms.Normalize(
        mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
        std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
    )
    
    n = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]  # Ensure 2D indexing
    
    for i, idx in enumerate(indices):
        image_tensor, label = dataset[idx]
        input_tensor = image_tensor.unsqueeze(0).to(device)
        
        # Generate Grad-CAM
        cam = grad_cam.generate(input_tensor)
        
        # Get prediction
        with torch.no_grad():
            prob = torch.sigmoid(model(input_tensor)).item()
        pred_class = 1 if prob > 0.5 else 0
        
        # Denormalize image for display
        display_img = inv_normalize(image_tensor).permute(1, 2, 0).numpy()
        display_img = np.clip(display_img, 0, 1)
        
        # Resize CAM to match image
        import cv2
        cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        
        # Original image
        axes[i, 0].imshow(display_img)
        axes[i, 0].set_title(f"True: {class_names[label]}", fontsize=11)
        axes[i, 0].axis("off")
        
        # Grad-CAM heatmap
        axes[i, 1].imshow(cam_resized, cmap="jet")
        axes[i, 1].set_title("Grad-CAM Heatmap", fontsize=11)
        axes[i, 1].axis("off")
        
        # Overlay
        axes[i, 2].imshow(display_img)
        axes[i, 2].imshow(cam_resized, cmap="jet", alpha=0.4)
        axes[i, 2].set_title(f"Pred: {class_names[pred_class]} ({prob:.2f})", fontsize=11)
        axes[i, 2].axis("off")
    
    plt.suptitle("Grad-CAM: Model Attention Regions", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Apply Grad-CAM to sample test images
# Target the last convolutional block of ResNet-50 (layer4)
target_layer = model.layer4[-1].conv3

# Pick a few normal and pneumonia samples from the test set
normal_indices = [i for i, (_, l) in enumerate(test_dataset.samples) if l == 0][:3]
pneumonia_indices = [i for i, (_, l) in enumerate(test_dataset.samples) if l == 1][:3]
sample_indices = normal_indices + pneumonia_indices

show_gradcam(model, test_dataset, sample_indices, target_layer, DEVICE, CLASS_NAMES)

---
## 9. (Optional) DenseNet-121 Comparison

DenseNet-121 is the backbone of **CheXNet** (Rajpurkar et al., 2017), which achieved radiologist-level performance on 14 chest X-ray pathologies. Here we provide the setup for a quick comparison.

In [ ]:
# Uncomment to train DenseNet-121 (adds ~10 min on GPU)
# densenet_model = build_densenet121(num_classes=1, freeze_backbone=False)
# densenet_model = densenet_model.to(DEVICE)
#
# densenet_optimizer = optim.AdamW(densenet_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# densenet_scheduler = optim.lr_scheduler.OneCycleLR(
#     densenet_optimizer, max_lr=LEARNING_RATE * 10,
#     steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS,
#     pct_start=0.3, anneal_strategy="cos"
# )
#
# print("Training DenseNet-121...")
# for epoch in range(1, NUM_EPOCHS + 1):
#     train_loss, train_acc = train_one_epoch(
#         densenet_model, train_loader, criterion, densenet_optimizer, densenet_scheduler, DEVICE
#     )
#     val_loss, val_acc, _, _ = evaluate(densenet_model, val_loader, criterion, DEVICE)
#     print(f"Epoch {epoch}/{NUM_EPOCHS} -- Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
#
# # Evaluate DenseNet on test set
# _, densenet_acc, densenet_probs, densenet_labels = evaluate(densenet_model, test_loader, criterion, DEVICE)
# densenet_auc = auc(*roc_curve(densenet_labels, densenet_probs)[:2])
# print(f"\nDenseNet-121 Test Accuracy: {densenet_acc:.4f}, AUC: {densenet_auc:.4f}")

print("DenseNet-121 training is commented out to save time.")
print("Uncomment the cell above to compare architectures.")

---
## 10. Summary & Discussion

### Results
- We trained a **ResNet-50** model fine-tuned on chest X-ray images for binary pneumonia detection.
- **WeightedRandomSampler** addressed the significant class imbalance in the training set.
- **OneCycleLR** scheduling helped achieve fast convergence.
- **Grad-CAM** visualizations confirmed the model attends to lung regions relevant to pneumonia diagnosis.

### Key Metrics for Medical AI
- **Sensitivity (Recall)** is paramount: missing a pneumonia case (false negative) can be life-threatening.
- **Specificity** matters too: false positives lead to unnecessary treatment and patient anxiety.
- **AUC** provides a threshold-independent measure of discriminative ability.

### Limitations
1. **Small validation set**: Only 16 images in the provided validation split, making validation metrics unreliable. In practice, consider merging val into train and using k-fold cross-validation.
2. **Binary classification only**: Real pneumonia comes in many forms (bacterial, viral, fungal). This model does not distinguish subtypes.
3. **Single institution data**: The dataset comes from one hospital, limiting generalizability across demographics, equipment, and imaging protocols.
4. **No external validation**: A robust clinical model requires testing on independent, multi-site datasets.

### Ethical Considerations in Medical AI
- **Not a diagnostic tool**: This model is a research prototype, not cleared for clinical use. Any clinical deployment requires rigorous validation and regulatory approval (e.g., FDA 510(k) or De Novo).
- **Bias and fairness**: Performance may vary across patient demographics (age, sex, ethnicity). Thorough subgroup analysis is essential.
- **Human-in-the-loop**: AI should augment, not replace, clinical decision-making. Models should flag cases for radiologist review, not make autonomous diagnoses.
- **Transparency**: Grad-CAM and similar techniques help clinicians understand model reasoning, building trust and enabling error detection.
- **Data privacy**: Medical imaging data is protected under regulations like HIPAA. Proper de-identification and consent are required.

### Next Steps
- Try **DenseNet-121** (uncomment Section 9) for comparison with the CheXNet architecture.
- Experiment with **larger image sizes** (e.g., 384x384) to capture fine-grained details.
- Apply **test-time augmentation (TTA)** for more robust predictions.
- Explore **ensemble methods** combining multiple architectures.
- Validate on external datasets (e.g., CheXpert, MIMIC-CXR, NIH ChestX-ray14).